In [ ]:
import napari
from tifffile import imread
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pims
import trackpy as tp

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# Load image

image = imread('/Volumes/gchao/bamfaile/Analysis/TUBB2B-KI/Batch20230223/D21/Denoised/Masks_created/100tp_561-100-50ms-1000g_4_conf561_merged.tif')
# In this image, ROI 9 shows detected spots that are high background rather than spots - can morphological appearance (eccentricity)
# of spot be used to do more filtering of bad quality / false-positive spots?

In [ ]:
print(image.dtype, image.shape, np.min(image), np.max(image))

In [ ]:
# Loads image mask

image_mask = imread('/Volumes/gchao/bamfaile/Analysis/TUBB2B-KI/Batch20230223/D21/ROIs/ROIs_as_mask_BIOP/100tp_561-100-50ms-1000g_4_conf561_merged_ROIs.tif')

In [ ]:
print(image_mask.dtype, image_mask.shape)

In [ ]:
# Initializes Napari viewer

viewer = napari.Viewer()

In [ ]:
# Adds image to Napari viewer

viewer.add_image(image)

In [ ]:
# Adds mask image to Napari viewer

viewer.add_image(image_mask,
                 colormap="gist_earth",
                 contrast_limits=[0,8],
                 opacity=0.2)

In [ ]:
# Loads detected spots

spots_path = r"/Volumes/gchao/bamfaile/Analysis/TUBB2B-KI/Batch20230223/D21/Spotdetection_tracking/Spotdetection/100tp_561-100-50ms-1000g_4_conf561_merged_spots.csv"
spots = pd.read_csv(spots_path)
spots.head()

In [ ]:
# Creates df that contains xyct axes in the same order as image --> tcxy

spots_tcyxcoord = spots.iloc[:, np.r_[0,3,2]]
spots_tcyxcoord.insert(1, "channel", 0)
spots_tcyxcoord

In [ ]:
# Creates array from df (necessary to use in the points layer of Napari)

spots_tcyxarray = spots_tcyxcoord.to_numpy()
spots_tcyxarray

In [ ]:
# Adds detected spots to points layer

viewer.add_points(spots_tcyxcoord,
                  face_color='#ffffff00',
                  edge_color='magenta')

In [ ]:
# Function to split table by ROI for tracking

def split_table_by_roi_id(df):
    """
    Splits each spots file into seperate dfs by roi-id, so that tracking can be done per ROI 
    """
    df = df.groupby("roi_id", sort = False, as_index = False)
    
    return df

In [ ]:
#Splits table by ROI

spots_per_ROI = [j for i,j in split_table_by_roi_id(spots)]
spots_per_ROI[0].head(10)

In [ ]:
# Links spots for all ROIs, filters them by minimum track length (default: 4 frames)
# Concatenates results of single ROIs into one df
# First number in df name stands for the distance in pixels that two spots can be apart, second number stands for how
# many frames a spot can be missing, before it will not be assigned to the same track anymore

tracks_per_ROI_5_3 = []

for df in spots_per_ROI:
    df = tp.link(df, 5, memory = 3)
    df = tp.filter_stubs(df, 4)
    tracks_per_ROI_5_3.append(df)

all_tracks_5_3 = pd.concat(tracks_per_ROI_5_3)

all_tracks_5_3.head(10)

In [ ]:
# Takes columns "particle", "frame", "x", "y" and sorts into format required by napari for visualization
# Inserts channel information right before y and x coordinate, according to image dimensions

all_tracks_5_3_sorted = all_tracks_5_3.iloc[:, np.r_[9,0,3,2]].sort_values("particle")
all_tracks_5_3_sorted.insert(2, "channel", 0)
all_tracks_5_3_sorted.head(20)

In [ ]:
# For single ROIs
# Takes columns "particle", "frame", "x", "y" and sorts into format required by napari for visualization
# Inserts channel information right before y and x coordinate, according to image dimensions

tracks_per_ROI_5_3_sorted = []

for df in tracks_per_ROI_5_3:
    df = df.iloc[:, np.r_[9,0,3,2]].sort_values("particle") 
    tracks_per_ROI_5_3_sorted.append(df)
tracks_per_ROI_5_3_sorted[0].head(20)

In [ ]:
# For single ROIs
# Inserts channel information right before y and x coordinate, according to image dimensions

for df in tracks_per_ROI_5_3_sorted:
    df = df.insert(2, "channel", 0)
tracks_per_ROI_5_3_sorted[0].head()


In [ ]:
# Creates array from df (necessary to use in the points layer of Napari)

tracks_per_ROI_5_3_array = []
for df in tracks_per_ROI_5_3_sorted:
    df = df.to_numpy()
    tracks_per_ROI_5_3_array.append(df)
tracks_per_ROI_5_3_array[0]

In [ ]:
# Adds tracks of all ROIs into Napari viewer as seperate track layers

for df in tracks_per_ROI_5_3_array:
    viewer.add_tracks(df, name='tracks')

In [ ]:
plt.figure()
tp.plot_traj(tracks_per_ROI_5_3_sorted[0]);